In [1]:
import asyncio
import json
import time
import math

import numpy as np
import pandas as pd

from collections import defaultdict, deque

import websockets

from datetime import datetime

In [2]:
SYMBOL = "btcusdt"

TRADE_LIMIT = 50000

DEPTH_LEVELS = 20

PROFILE_TICK = 10

FOOTPRINT_WINDOW = 60

ABSORPTION_THRESHOLD = 50

ICEBERG_REPEAT = 5

SIGNAL_THRESHOLD = 80

In [3]:
trades = deque(maxlen=50000)

depth_snapshot = {
    "bids": {},
    "asks": {}
}

book_ticker = {}

cvd_history = []

footprint = defaultdict(
    lambda:{
        "buy":0,
        "sell":0
    }
)

volume_profile = defaultdict(float)

signals = deque(maxlen=100)

market_state = {}

In [4]:
TRADE_WS = f"wss://fstream.binance.com/ws/{SYMBOL}@aggTrade"

DEPTH_WS = f"wss://fstream.binance.com/ws/{SYMBOL}@depth20@100ms"

BOOK_WS = f"wss://fstream.binance.com/ws/{SYMBOL}@bookTicker"

In [5]:
def process_depth(msg):

    depth_snapshot["bids"] = {
        float(p): float(q)
        for p,q in msg["b"]
    }

    depth_snapshot["asks"] = {
        float(p): float(q)
        for p,q in msg["a"]
    }

In [6]:
async def trade_stream():

    async with websockets.connect(
        TRADE_WS
    ) as ws:

        while True:

            msg = json.loads(
                await ws.recv()
            )

            process_trade(msg)

In [7]:
async def depth_stream():

    async with websockets.connect(
        DEPTH_WS
    ) as ws:

        while True:

            msg = json.loads(
                await ws.recv()
            )

            process_depth(msg)

In [8]:
async def book_stream():

    async with websockets.connect(
        BOOK_WS
    ) as ws:

        while True:

            msg = json.loads(
                await ws.recv()
            )

            process_book(msg)

In [9]:
def process_trade(msg):

    price = float(msg["p"])

    qty = float(msg["q"])

    ts = msg["T"]

    is_sell = msg["m"]

    trades.append({

        "price":price,

        "qty":qty,

        "sell":is_sell,

        "time":ts

    })

    update_cvd(price,qty,is_sell)

    update_footprint(price,qty,is_sell)

    update_profile(price,qty)

In [10]:
def process_depth(msg):

    depth_snapshot["bids"] = {
        float(p): float(q)
        for p,q in msg["b"]
    }

    depth_snapshot["asks"] = {
        float(p): float(q)
        for p,q in msg["a"]
    }

In [11]:
def process_book(msg):

    book_ticker["bid"] = float(msg["b"])
    book_ticker["ask"] = float(msg["a"])

    book_ticker["bid_qty"] = float(msg["B"])
    book_ticker["ask_qty"] = float(msg["A"])

In [12]:
current_cvd = 0

In [13]:
def update_cvd(price,qty,is_sell):

    global current_cvd

    if is_sell:

        current_cvd -= qty

    else:

        current_cvd += qty

    cvd_history.append({

        "time":time.time(),

        "cvd":current_cvd,

        "price":price

    })

In [14]:
def update_footprint(
    price,
    qty,
    is_sell
):

    level = round(price)

    if is_sell:

        footprint[level]["sell"] += qty

    else:

        footprint[level]["buy"] += qty

In [15]:
def update_profile(
    price,
    qty
):

    bucket = (
        round(
            price /
            PROFILE_TICK
        )
        *
        PROFILE_TICK
    )

    volume_profile[bucket] += qty

In [16]:
def get_poc():

    if not volume_profile:

        return None

    return max(
        volume_profile,
        key=volume_profile.get
    )

In [17]:
def value_area():

    total = sum(
        volume_profile.values()
    )

    target = total * 0.7

    sorted_levels = sorted(
        volume_profile.items(),
        key=lambda x:x[1],
        reverse=True
    )

    running = 0

    levels = []

    for p,v in sorted_levels:

        levels.append(p)

        running += v

        if running >= target:
            break

    return min(levels),max(levels)

In [18]:
def liquidity_clusters():

    bids = depth_snapshot["bids"]

    asks = depth_snapshot["asks"]
    if len(bids)==0 or len(asks)==0:
        return []

    bid_mean = np.mean(
        list(bids.values())
    )

    ask_mean = np.mean(
        list(asks.values())
    )

    clusters = []

    for p,q in bids.items():

        if q > bid_mean*3:

            clusters.append(
                ("BID",p,q)
            )

    for p,q in asks.items():

        if q > ask_mean*3:

            clusters.append(
                ("ASK",p,q)
            )

    return clusters

In [19]:
def imbalance():

    bid_vol = sum(
        depth_snapshot["bids"].values()
    )

    ask_vol = sum(
        depth_snapshot["asks"].values()
    )

    if bid_vol+ask_vol == 0:

        return 0

    return (

        bid_vol - ask_vol

    ) / (

        bid_vol + ask_vol

    )

In [20]:
def detect_absorption():

    recent = list(trades)[-200:]

    if len(recent)<100:

        return None

    sell_volume = sum(
        x["qty"]
        for x in recent
        if x["sell"]
    )

    prices = [
        x["price"]
        for x in recent
    ]

    move = max(prices)-min(prices)

    if (
        sell_volume >
        ABSORPTION_THRESHOLD
        and move < 20
    ):
        return "BULLISH"

    buy_volume = sum(
        x["qty"]
        for x in recent
        if not x["sell"]
    )

    if (
        buy_volume >
        ABSORPTION_THRESHOLD
        and move < 20
    ):
        return "BEARISH"

    return None

In [21]:
def detect_iceberg():

    repeated = defaultdict(int)

    recent = list(trades)[-500:]

    for t in recent:

        price = round(
            t["price"]
        )

        repeated[price]+=1

    for p,c in repeated.items():

        if c >= ICEBERG_REPEAT:

            return p

    return None

In [22]:
def market_structure():

    recent = list(trades)[-500:]

    if len(recent) == 0:

        return {
            "high": None,
            "low": None,
            "price": None
        }

    prices = [
        x["price"]
        for x in recent
    ]

    return {
        "high": max(prices),
        "low": min(prices),
        "price": prices[-1]
    }

In [23]:
def build_signal():

    score = 0

    reason = []
    if len(cvd_history)>50:

        if (

            cvd_history[-1]["cvd"]

            >

            cvd_history[-50]["cvd"]

        ):

            score += 20

            reason.append(
            "CVD_UP"
            )

        else:

            score -= 20

    imb = imbalance()

    if imb > 0.3:

        score += 15

        reason.append(
        "BID_DOMINANT"
         )

    if imb < -0.3:

        score -= 15


    absorb = detect_absorption()

    if absorb == "BULLISH":

        score += 25

    if absorb == "BEARISH":

        score -= 25

    poc = get_poc()

    ms = market_structure()

    if ms["price"] is None:
        return "WAIT",0,["NO_DATA"]

    price = ms["price"]

    if poc is not None:

        if price > poc:
            score += 10
        else:
            score -= 10

    level = round(price)

    fp = footprint[level]

    delta = (
        fp["buy"]
        -
        fp["sell"]
    )

    if delta > 0:

        score += 15

    else:

        score -= 15

    if score >= 80:

        return "LONG",score,reason

    if score <= -80:

        return "SHORT",score,reason

    return "WAIT",score,reason

In [24]:
async def dashboard():

    while True:
        if len(trades) < 50:

            print(
                "Collecting data..."
            )

            await asyncio.sleep(1)

            continue
            

        signal = build_signal()

        print(
               "\n"*5
         )

        print(
            "="*60
        )

        print(
            datetime.now()
        )

        ms = market_structure()

        print(
            "PRICE:",
            ms["price"]
        )

        print(
            "CVD:",
            current_cvd
        )

        print(
            "POC:",
            get_poc()
        )

        print(
            "IMBALANCE:",
            imbalance()
        )
 
        print(
            "SIGNAL:",
            signal
        )

        await asyncio.sleep(1)

In [25]:
import asyncio
import websockets
import json

async def test_connection():
    try:
        async with websockets.connect("wss://fstream.binance.com/ws/btcusdt@aggTrade") as ws:
            print("CONNECTED! Waiting for message...")
            msg = await asyncio.wait_for(ws.recv(), timeout=10)
            print(f"RECEIVED: {msg[:200]}")
    except Exception as e:
        print(f"FAILED: {e}")

await test_connection()

CONNECTED! Waiting for message...
FAILED: 


In [26]:
await asyncio.gather(

    trade_stream(),

    depth_stream(),

    book_stream(),

    dashboard()

)

CancelledError: 